# Issue #47 — Resolution-aware training diagnostic

Two matched short runs test whether training at 960 adds benefit beyond inference resizing. This is not a v3 preprocessing policy or S20/development evidence.

**Inputs:** Committed training declaration and compact saved evidence. Acquisition requires pinned YOLO26n/CUDA, original annotation metadata and existing Raw training / consumed Skip evaluation exports.

**Outputs:** Read-only 2×2 matrix, source-group uncertainty, class/cohort errors, cost and decision; opted-in acquisition writes only new docs/issue47-resolution-training and runs/issue47-resolution-training/study artifacts.

**Prerequisites:** Read docs/ISSUE47_TRAINING_PLAN.md. No upstream notebook is executed. Existing/incomplete attempts block re-execution. Confirmation, development and final pixels stay sealed.

## 1. Configuration

Default execution verifies compact committed artifacts without models, annotations, datasets or local runs. Commit the declaration and implementation before enabling acquisition.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from vehicle_detection.common import require_paths
from vehicle_detection.resolution_training import run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
require_paths(plan=ROOT / "docs/ISSUE47_TRAINING_PLAN.md")
ALLOW_TRAINING = False

## 2. Fixed paired acquisition or saved evidence

Whole-group S20 prefix; five epochs at each training size, identical batch 16/settings, independently initialized COCO checkpoint. Report fixed last.pt. Only the consumed 1,027 exploration images are evaluated. This short study does not select checkpoints on development or establish convergence.

In [ ]:
result = run(ROOT, allow_training=ALLOW_TRAINING)
print(result["population"])
rows = []
for key, metrics in result["metrics"].items():
    train_size, inference_size = key.split("/")
    rows.append(
        {
            "train_size": train_size,
            "inference_size": inference_size,
            "mAP50:95 (%)": 100 * metrics["all"]["mAP50_95"],
            "mAP50 (%)": 100 * metrics["all"]["mAP50"],
        }
    )
display(pd.DataFrame(rows))
display(Image(filename=str(ROOT / "docs/issue47-resolution-training/training.png")))

## 3. Incremental training benefit and uncertainty

Primary contrast: 960-trained versus 640-trained, both evaluated at 960. Bootstrap uses original source groups, 100 fixed paired draws, all-five-class macro and no missing-class imputation. Conditional intervals are exploratory on consumed data. Rare-class uncertainty is reported separately.

In [ ]:
interval = result["paired_interval"]
print("Primary incremental AP points:", 100 * interval["recovery"])
print(
    "Conditional 95% interval:",
    [100 * interval[k] for k in ("recovery_ci_low", "recovery_ci_high")],
)
print("Valid / excluded draws:", interval["valid_draws"], interval["invalid_draws"])
display(pd.DataFrame(interval["per_class"]))
for cohort in ("all", "dark", "bright", "low_contrast", "low_sharpness", "small_two_wheeler"):
    print(cohort, "— all original GT retained; cohort AP is not AP-small")
    for arm in ("640/960", "960/960"):
        print(arm)
        display(pd.DataFrame(result["metrics"][arm][cohort]["classes"]).T)
        display(pd.DataFrame(result["metrics"][arm][cohort]["counts"]).T)

## 4. Cost and illustrative errors

Training wall times and peak allocated VRAM are measured per run. Native inference means and synchronized batch p95 include sequential/setup effects; batch p95 is not image p95 or deployment latency. There is no added offline filter. Examples rank TP change, FP change, then original ID; no individual AP or missed-box attribution is inferred.

In [ ]:
display(pd.DataFrame(result["training"]).T)
display(pd.DataFrame(result["timings"]).T)
display(pd.DataFrame(result["examples"]))

## 5. Summary and next boundary

Preserve every failed prospective gate. A useful signal recommends a later separately declared group-disjoint study; it cannot select an adaptive policy or promote S20. One seed and a short training budget do not establish convergence or generalization. Existing Gamma/Sharpen and 640/960 pretrained evidence remain unchanged. Cross-version S20 results belong only in experiments/compare.ipynb.

In [ ]:
display(result["decision"])
assert not result["decision"]["policy_selected"]
assert not result["decision"]["s20_promoted"]
assert not result["confirmation_development_final_pixels_accessed"]
print("Issue #47 remains open; no confirmation/development/final inference or full S20 promotion.")